In [1]:
# ---- setup -----------------------------------------------------------------
import os, sys, glob, json, time
from pathlib import Path

CODE_DIR = Path("/kaggle/working/code"); CODE_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR = sorted(d for d in glob.glob("/kaggle/input/**/m4raw_cache", recursive=True)
                   if os.path.isdir(os.path.join(d, "train")))[0]
print("cache:", CACHE_DIR)
print(json.loads(Path(CACHE_DIR, "manifest.json").read_text())["splits"])

cache: /kaggle/input/datasets/mahimasunilbachhav/m4raw-sgf-cache/m4raw_cache
{'train': {'pair_volumes': 384, 'slices': 6912, 'subjects': 128}, 'val': {'pair_volumes': 90, 'slices': 1620, 'subjects': 30}}


In [2]:
%%writefile /kaggle/working/code/sgf_train.py
"""
Week-2 toolkit for reproducing SGF-MRI on M4Raw (T1 reference -> T2 target).

Contents
  * models      : Bicubic / zero-filled baseline, single- and multi-contrast U-Net
  * train()     : L1 loss, Adam 2e-4, StepLR(40, 0.1), batch 4  (SGF-MRI Sec. 4.1)
                  checkpoint every epoch, auto-resume, stops cleanly before Kaggle's 12 h limit
  * evaluate()  : PSNR / SSIM per volume, exactly as DCAMSR/fastMRI compute them
  * profile()   : parameters, peak GPU memory, inference time per slice

Every model uses DCAMSR's call signature  model(lr, ref, ref_down)
    lr       (B,1,h,w)  degraded T2 (LR image for SR, zero-filled image for reconstruction)
    ref      (B,1,H,W)  fully sampled T1
    ref_down (B,1,h,w)  T1 degraded the same way as the input (SR only)
and returns (B,1,H,W).  DCAMSR and SGF-MRI will plug in later with no other changes.
"""
from __future__ import annotations

import csv
import glob
import json
import math
import os
import random
import shutil
import time
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Optional

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

from m4raw_dataset import M4RawPairs
from m4raw_prep import fft2c, ifft2c


# =========================================================================== #
# Locating the cache on Kaggle
# =========================================================================== #
def find_cache_dir() -> str:
    hits = sorted(glob.glob("/kaggle/input/**/m4raw_cache", recursive=True))
    hits = [h for h in hits if os.path.isdir(os.path.join(h, "train"))]
    if not hits:
        raise FileNotFoundError("m4raw_cache not found: Add Input -> Your Datasets -> your cache dataset")
    return hits[0]


# =========================================================================== #
# Dataset: same as M4RawPairs, plus a choice of which axis the recon mask drops
# =========================================================================== #
class M4RawTask(M4RawPairs):
    """
    mask_axis = -1 : drop k-space columns (fastMRI convention, default in m4raw_dataset)
    mask_axis = -2 : drop k-space rows    (the SGF-MRI paper text says "rows")
    """

    def __init__(self, *a, mask_axis: int = -1, **kw):
        super().__init__(*a, **kw)
        assert mask_axis in (-1, -2)
        self.mask_axis = mask_axis

    def _zero_filled(self, img, name):
        from m4raw_dataset import _seed_from_name
        seed = _seed_from_name(name) if self.deterministic else None
        n = img.shape[self.mask_axis]
        mask = self.mask_func(n, seed)
        m2d = mask[None, :] if self.mask_axis == -1 else mask[:, None]
        k = fft2c(img.astype(np.complex64)) * m2d
        return np.abs(ifft2c(k)).astype(np.float32), mask


def worker_init_fn(_):
    """Give every DataLoader worker its own random-mask stream.
    Without this, all workers start from the same copied RandomState and
    produce identical 'random' masks (a classic numpy + DataLoader bug)."""
    info = torch.utils.data.get_worker_info()
    ds = info.dataset
    seed = info.seed % 2**32
    np.random.seed(seed)
    random.seed(seed)
    if hasattr(ds, "mask_func"):
        ds.mask_func.rng = np.random.RandomState(seed)


def make_dataset(cache_dir, split, cfg: "Config", max_pairs=None):
    return M4RawTask(cache_dir, split, task=cfg.task, scale=cfg.scale, accel=cfg.accel,
                     mask_axis=cfg.mask_axis, max_pairs=max_pairs)


# =========================================================================== #
# Models
# =========================================================================== #
def upsample_to(x, hw, mode="bicubic"):
    if tuple(x.shape[-2:]) == tuple(hw):
        return x
    return F.interpolate(x, size=hw, mode=mode, align_corners=False)


class Bicubic(nn.Module):
    """
    No learning. SR: bicubic upsampling of the LR image. Recon: the zero-filled image itself.

    DCAMSR makes LR images by cropping the centre of k-space with an orthonormal FFT,
    which leaves them `scale` times BRIGHTER than the HR image (2x / 4x). Trained
    networks learn this (DCAMSR and SGF-MRI were trained on exactly this data), but a
    plain interpolation cannot, so evaluate() divides this model's de-normalised output
    by `intensity_divisor`.
    """

    def __init__(self, scale: int = 1, **_):
        super().__init__()
        self.intensity_divisor = float(scale)
        self.dummy = nn.Parameter(torch.zeros(()), requires_grad=False)

    def forward(self, lr, ref, ref_down=None):
        return upsample_to(lr, ref.shape[-2:])


def _norm_layer(c, norm):
    return nn.InstanceNorm2d(c) if norm == "instance" else nn.Identity()


class ConvBlock(nn.Module):
    def __init__(self, cin, cout, norm="none"):
        super().__init__()
        bias = norm == "none"
        self.net = nn.Sequential(
            nn.Conv2d(cin, cout, 3, padding=1, bias=bias), _norm_layer(cout, norm), nn.LeakyReLU(0.2, True),
            nn.Conv2d(cout, cout, 3, padding=1, bias=bias), _norm_layer(cout, norm), nn.LeakyReLU(0.2, True))

    def forward(self, x):
        return self.net(x)


class UNet(nn.Module):
    """
    fastMRI-style U-Net (conv-LeakyReLU x2, avg-pool down, transpose-conv up).
    norm="none" by default: with DCAMSR's data the model must recover each image's
    absolute brightness level (LR is `scale`x brighter than HR), and InstanceNorm
    erases exactly that information (tested: single-contrast SR stalls with it).
    """

    def __init__(self, in_chans=1, out_chans=1, chans=32, num_pools=3, norm="none"):
        super().__init__()
        self.num_pools = num_pools
        self.down = nn.ModuleList([ConvBlock(in_chans, chans, norm)])
        c = chans
        for _ in range(num_pools - 1):
            self.down.append(ConvBlock(c, c * 2, norm)); c *= 2
        self.bottom = ConvBlock(c, c * 2, norm)
        self.up_t, self.up_c = nn.ModuleList(), nn.ModuleList()
        for _ in range(num_pools):
            self.up_t.append(nn.Sequential(nn.ConvTranspose2d(c * 2, c, 2, stride=2, bias=norm == "none"),
                                           _norm_layer(c, norm), nn.LeakyReLU(0.2, True)))
            self.up_c.append(ConvBlock(c * 2, c, norm)); c //= 2
        self.out = nn.Conv2d(chans, out_chans, 1)

    def forward(self, x):
        h, w = x.shape[-2:]
        m = 2 ** self.num_pools                      # pad so every pooling divides evenly
        ph, pw = (-h) % m, (-w) % m
        x = F.pad(x, (pw // 2, pw - pw // 2, ph // 2, ph - ph // 2), mode="reflect")
        skips = []
        for blk in self.down:
            x = blk(x); skips.append(x); x = F.avg_pool2d(x, 2)
        x = self.bottom(x)
        for t, c in zip(self.up_t, self.up_c):
            x = c(torch.cat([t(x), skips.pop()], 1))
        x = self.out(x)
        return x[..., ph // 2: ph // 2 + h, pw // 2: pw // 2 + w]


class UNetRestorer(nn.Module):
    """
    Upsample the degraded T2 to full size, then predict a residual correction.
      use_ref=False : single-contrast baseline (T2 only)
      use_ref=True  : plain multi-contrast baseline (T1 concatenated as a 2nd channel)
    The pair (single vs multi) is the "does the reference help?" first row of Fig. 5a.
    """

    def __init__(self, use_ref=False, chans=32, num_pools=3, norm="none", **_):
        super().__init__()
        self.use_ref = use_ref
        self.unet = UNet(in_chans=2 if use_ref else 1, chans=chans, num_pools=num_pools, norm=norm)

    def forward(self, lr, ref, ref_down=None):
        up = upsample_to(lr, ref.shape[-2:])
        inp = torch.cat([up, ref], 1) if self.use_ref else up
        return up + self.unet(inp)


MODELS = {
    "bicubic": lambda cfg: Bicubic(cfg.scale if cfg.task == "sr" else 1),
    "unet_single": lambda cfg: UNetRestorer(False, cfg.chans, cfg.num_pools, cfg.norm),
    "unet_multi": lambda cfg: UNetRestorer(True, cfg.chans, cfg.num_pools, cfg.norm),
    # Week 3: "dcamsr": ..., Week 4-5: "sgf_mri": ...
}


def build_model(cfg: "Config") -> nn.Module:
    if cfg.model not in MODELS:
        raise KeyError(f"unknown model {cfg.model!r}; choose from {list(MODELS)}")
    return MODELS[cfg.model](cfg)


def count_params(model) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


# =========================================================================== #
# Config
# =========================================================================== #
@dataclass
class Config:
    run_name: str = "unet_single_sr4"
    model: str = "unet_single"
    task: str = "sr"                 # "sr" or "recon"
    scale: int = 4                   # SR factor (2 or 4)
    accel: int = 4                   # recon acceleration (4 or 8)
    mask_axis: int = -1              # recon: -1 columns (fastMRI) / -2 rows (paper text)
    chans: int = 32
    num_pools: int = 3               # paper: "encoder-decoder has 3 layers"
    norm: str = "none"               # U-Net normalisation: "none" | "instance"
    epochs: int = 50                 # not given in the paper; DCAMSR's train.py uses 50
    batch_size: int = 4              # paper
    lr: float = 2e-4                 # paper
    lr_step: int = 40                # paper
    lr_gamma: float = 0.1            # paper
    num_workers: int = 4
    amp: bool = True                 # mixed precision on GPU (~2x faster, less memory)
    multi_gpu: bool = False          # Kaggle "GPU T4 x2": DataParallel over both cards
    seed: int = 42
    val_every: int = 5               # full test-set evaluation every N epochs (+ last epoch)
    time_budget_h: float = 11.0      # stop cleanly before Kaggle's 12 h session limit
    max_train_pairs: Optional[int] = None   # small numbers = quick smoke test
    max_val_pairs: Optional[int] = None
    out_root: str = "/kaggle/working/runs"

    @property
    def out_dir(self) -> Path:
        return Path(self.out_root) / self.run_name

    def describe(self):
        deg = f"SR x{self.scale}" if self.task == "sr" else f"recon {self.accel}x (mask axis {self.mask_axis})"
        return f"{self.model} | {deg}"


# =========================================================================== #
# Evaluation (DCAMSR / fastMRI protocol)
# =========================================================================== #
def _volume_scores(gt, pred):
    """fastmri/evaluate.py: data_range = volume max; SSIM averaged over slices."""
    from skimage.metrics import peak_signal_noise_ratio, structural_similarity
    dr = float(gt.max())
    p = peak_signal_noise_ratio(gt, pred, data_range=dr)
    s = float(np.mean([structural_similarity(g, q, data_range=dr) for g, q in zip(gt, pred)]))
    return p, s


@torch.no_grad()
def evaluate(model, dataset, device, batch_size=8, num_workers=4, amp=True, keep_examples=0):
    """
    Runs the whole split in order, rebuilds each volume, de-normalises with
    x * std + mean (the GT is the de-normalised, [-6,6]-clipped target, exactly
    what DCAMSR's validation_step compares against), and scores per volume.
    Returns metrics dict (+ a list of example slices if keep_examples > 0).
    """
    model.eval()
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False,
                        num_workers=num_workers, pin_memory=device.type == "cuda")
    use_amp = amp and device.type == "cuda"
    n_slices = {}
    for path, s in dataset.examples:
        n_slices[path.stem] = n_slices.get(path.stem, 0) + 1
    psnrs, ssims, names, examples = [], [], [], []
    cur, preds, gts = None, [], []

    def flush():
        if cur is None:
            return
        p, s = _volume_scores(np.stack(gts), np.stack(preds))
        psnrs.append(p); ssims.append(s); names.append(cur)

    for b in loader:
        lr = b["input"].unsqueeze(1).to(device)
        ref = b["ref"].unsqueeze(1).to(device)
        ref_in = b["ref_in"].unsqueeze(1).to(device)
        with torch.autocast("cuda", dtype=torch.float16, enabled=use_amp):
            out = model(lr, ref, ref_in)
        out = out.float().squeeze(1).cpu().numpy()
        mean = b["mean"].numpy()[:, None, None]
        std = b["std"].numpy()[:, None, None]
        out = (out * std + mean) / getattr(model, "intensity_divisor", 1.0)
        tgt = b["target"].numpy() * std + mean
        for i, name in enumerate(b["fname"]):
            if name != cur:
                flush(); cur, preds, gts = name, [], []
            preds.append(out[i]); gts.append(tgt[i])
            if len(examples) < keep_examples and int(b["slice"][i]) == n_slices[name] // 2:
                examples.append({"name": name, "input": b["input"][i].numpy() * std[i] + mean[i],
                                 "output": out[i], "target": tgt[i], "ref": b["ref"][i].numpy()})
    flush()
    res = {"psnr": float(np.mean(psnrs)), "psnr_std": float(np.std(psnrs)),
           "ssim": float(np.mean(ssims)), "ssim_std": float(np.std(ssims)),
           "n_volumes": len(psnrs)}
    return (res, examples) if keep_examples else res


# =========================================================================== #
# Checkpointing
# =========================================================================== #
def _save(state, path: Path):
    tmp = path.with_suffix(".tmp")
    torch.save(state, tmp)
    os.replace(tmp, path)                     # atomic: a crash never leaves a broken file


def _bring_previous_run(cfg: Config):
    """If this run is not in /kaggle/working, copy it from an attached previous version."""
    if (cfg.out_dir / "last.pt").exists():
        return
    hits = sorted(glob.glob(f"/kaggle/input/**/runs/{cfg.run_name}/last.pt", recursive=True))
    if hits:
        src = Path(hits[-1]).parent
        shutil.copytree(src, cfg.out_dir, dirs_exist_ok=True)
        print(f"[resume] copied previous run from {src}")


# =========================================================================== #
# Training
# =========================================================================== #
def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)


def train(cfg: Config, cache_dir: str):
    t_start = time.time()
    seed_everything(cfg.seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    cfg.out_dir.mkdir(parents=True, exist_ok=True)
    _bring_previous_run(cfg)

    train_ds = make_dataset(cache_dir, "train", cfg, cfg.max_train_pairs)
    val_ds = make_dataset(cache_dir, "val", cfg, cfg.max_val_pairs)
    g = torch.Generator(); g.manual_seed(cfg.seed)
    train_dl = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, drop_last=True,
                          num_workers=cfg.num_workers, pin_memory=device.type == "cuda",
                          worker_init_fn=worker_init_fn, generator=g,
                          persistent_workers=cfg.num_workers > 0)

    model = build_model(cfg).to(device)
    n_params = count_params(model)
    net = nn.DataParallel(model) if (cfg.multi_gpu and torch.cuda.device_count() > 1) else model
    opt = torch.optim.Adam(model.parameters(), lr=cfg.lr, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.0)
    sched = torch.optim.lr_scheduler.StepLR(opt, cfg.lr_step, cfg.lr_gamma)
    use_amp = cfg.amp and device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    start_epoch, best, history = 0, -math.inf, []
    last = cfg.out_dir / "last.pt"
    if last.exists():
        ck = torch.load(last, map_location=device, weights_only=False)
        model.load_state_dict(ck["model"]); opt.load_state_dict(ck["optim"])
        sched.load_state_dict(ck["sched"]); scaler.load_state_dict(ck["scaler"])
        start_epoch, best, history = ck["epoch"] + 1, ck["best_psnr"], ck["history"]
        print(f"[resume] continuing from epoch {start_epoch} (best PSNR so far {best:.3f})")
    (cfg.out_dir / "config.json").write_text(json.dumps(asdict(cfg), indent=2))

    print(f"{cfg.describe()} | params {n_params/1e6:.2f} M | device {device}"
          f"{' x' + str(torch.cuda.device_count()) if isinstance(net, nn.DataParallel) else ''} | "
          f"train {len(train_ds)} slices ({len(train_dl)} it/epoch) | test {len(val_ds)} slices")
    if start_epoch >= cfg.epochs:
        print("already finished"); return history

    epoch_times = []
    for epoch in range(start_epoch, cfg.epochs):
        t0 = time.time()
        net.train()
        tot, n = 0.0, 0
        for it, b in enumerate(train_dl):
            lr_img = b["input"].unsqueeze(1).to(device, non_blocking=True)
            ref = b["ref"].unsqueeze(1).to(device, non_blocking=True)
            ref_in = b["ref_in"].unsqueeze(1).to(device, non_blocking=True)
            tgt = b["target"].unsqueeze(1).to(device, non_blocking=True)
            with torch.autocast("cuda", dtype=torch.float16, enabled=use_amp):
                out = net(lr_img, ref, ref_in)
            loss = F.l1_loss(out.float(), tgt)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update()
            if not torch.isfinite(loss):
                raise FloatingPointError(f"loss became {loss.item()} at epoch {epoch} iter {it}")
            tot += loss.item(); n += 1
            if it % 500 == 0:
                print(f"  epoch {epoch:3d} iter {it:5d}/{len(train_dl)} loss {loss.item():.4f}", flush=True)
        lr_used = opt.param_groups[0]["lr"]
        sched.step()
        rec = {"epoch": epoch, "lr": lr_used, "train_l1": tot / max(n, 1),
               "train_min": (time.time() - t0) / 60}

        do_val = (epoch + 1) % cfg.val_every == 0 or epoch + 1 == cfg.epochs
        if do_val:
            tv = time.time()
            m = evaluate(model, val_ds, device, num_workers=cfg.num_workers, amp=cfg.amp)
            rec.update({k: m[k] for k in ("psnr", "psnr_std", "ssim", "ssim_std")})
            rec["val_min"] = (time.time() - tv) / 60
            if m["psnr"] > best:
                best = m["psnr"]
                _save({"model": model.state_dict(), "epoch": epoch, "metrics": m, "cfg": asdict(cfg)},
                      cfg.out_dir / "best.pt")
        history.append(rec)
        _save({"model": model.state_dict(), "optim": opt.state_dict(), "sched": sched.state_dict(),
               "scaler": scaler.state_dict(), "epoch": epoch, "best_psnr": best,
               "history": history, "cfg": asdict(cfg)}, last)
        _write_log(history, cfg.out_dir / "log.csv")

        epoch_times.append(time.time() - t0)
        msg = f"epoch {epoch:3d} | L1 {rec['train_l1']:.4f} | {epoch_times[-1]/60:.1f} min"
        if do_val:
            msg += f" | test PSNR {rec['psnr']:.2f} SSIM {rec['ssim']:.4f} (best {best:.2f})"
        print(msg, flush=True)

        elapsed = time.time() - t_start
        if epoch + 1 < cfg.epochs and elapsed + 1.5 * max(epoch_times[-3:]) > cfg.time_budget_h * 3600:
            print(f"[time budget] stopping after epoch {epoch} ({elapsed/3600:.1f} h). "
                  f"Run again (attach this version's output as input) to resume.")
            break
    return history


def _write_log(history, path):
    keys = sorted({k for r in history for k in r}, key=lambda k: (k != "epoch", k))
    with open(path, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=keys); w.writeheader(); w.writerows(history)


def load_for_eval(cfg: Config, which="best", device=None):
    device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = build_model(cfg).to(device)
    ck = torch.load(cfg.out_dir / f"{which}.pt", map_location=device, weights_only=False)
    model.load_state_dict(ck["model"])
    return model, ck.get("epoch")


# =========================================================================== #
# Efficiency (paper Sec. 4.4: memory / time / parameters)
# =========================================================================== #
@torch.no_grad()
def profile(model, cfg: Config, hw=(256, 256), repeats=50):
    device = next(model.parameters()).device
    H, W = hw
    h, w = (H // cfg.scale, W // cfg.scale) if cfg.task == "sr" else (H, W)
    lr = torch.randn(1, 1, h, w, device=device)
    ref = torch.randn(1, 1, H, W, device=device)
    ref_in = torch.randn(1, 1, h, w, device=device)
    model.eval()
    for _ in range(5):
        model(lr, ref, ref_in)
    if device.type == "cuda":
        torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats()
    t = time.time()
    for _ in range(repeats):
        model(lr, ref, ref_in)
    if device.type == "cuda":
        torch.cuda.synchronize()
    out = {"params_M": count_params(model) / 1e6, "sec_per_slice": (time.time() - t) / repeats}
    if device.type == "cuda":
        out["peak_mem_MB"] = torch.cuda.max_memory_allocated() / 2**20
    return out


# =========================================================================== #
# Paper numbers (M4Raw rows of Tables 1 and 2) for side-by-side comparison
# =========================================================================== #
PAPER_M4RAW = {
    ("sr", 2):    {"SwinIR": (32.16, 0.777), "DCAMSR": (32.31, 0.779), "SGF-MRI": (32.42, 0.781)},
    ("sr", 4):    {"SwinIR": (29.73, 0.709), "DCAMSR": (30.48, 0.728), "SGF-MRI": (30.62, 0.731)},
    ("recon", 4): {"SwinIR": (29.12, 0.692), "DCAMSR": (29.62, 0.706), "SGF-MRI": (29.82, 0.716)},
    ("recon", 8): {"SwinIR": (27.93, 0.621), "DCAMSR": (28.09, 0.658), "SGF-MRI": (28.27, 0.666)},
}


def paper_key(cfg: Config):
    return (cfg.task, cfg.scale if cfg.task == "sr" else cfg.accel)


def append_result(row: dict, path="/kaggle/working/results.csv"):
    path = Path(path)
    rows = []
    if path.exists():
        with open(path) as f:
            rows = list(csv.DictReader(f))
    rows = [r for r in rows if not (r.get("run") == row["run"] and r.get("checkpoint") == row.get("checkpoint"))]
    rows.append({k: (f"{v:.4f}" if isinstance(v, float) else v) for k, v in row.items()})
    keys = list(dict.fromkeys(k for r in rows for k in r))
    with open(path, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=keys); w.writeheader(); w.writerows(rows)

Writing /kaggle/working/code/sgf_train.py


In [3]:
sys.path.insert(0, CACHE_DIR)          # m4raw_dataset.py / m4raw_prep.py from the cache dataset
sys.path.insert(0, str(CODE_DIR))      # sgf_train.py
import importlib, torch, numpy as np, pandas as pd
import sgf_train as T; importlib.reload(T)
from sgf_train import Config, train, evaluate, load_for_eval, profile, append_result, PAPER_M4RAW, paper_key

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| device:", DEVICE,
      *(torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())))
NUM_WORKERS = min(4, os.cpu_count())

torch 2.11.0+cu128 | device: cuda Tesla T4 Tesla T4


In [4]:
DO_BASELINES = True
DO_SMOKE     = True
DO_TRAIN     = False
DO_EVAL      = True      # evaluates the run below if a checkpoint exists

cfg = Config(
    run_name    = "unet_single_sr4",
    model       = "unet_single",   # "unet_single" (T2 only) | "unet_multi" (T2 + T1 reference)
    task        = "sr",            # "sr" | "recon"
    scale       = 4,               # SR: 2 | 4
    accel       = 4,               # recon: 4 | 8
    mask_axis   = -1,              # recon: -1 drop columns (fastMRI) | -2 drop rows (paper wording)
    epochs      = 50,
    val_every   = 5,
    num_workers = NUM_WORKERS,
    multi_gpu   = torch.cuda.device_count() > 1,
)
print(cfg.describe(), "->", cfg.out_dir)

unet_single | SR x4 -> /kaggle/working/runs/unet_single_sr4


In [5]:
if DO_BASELINES:
    rows = []
    for task, f in [("sr", 2), ("sr", 4), ("recon", 4), ("recon", 8)]:
        c = Config(model="bicubic", task=task, scale=f, accel=f, mask_axis=cfg.mask_axis)
        ds = T.make_dataset(CACHE_DIR, "val", c)
        t0 = time.time()
        m = evaluate(T.build_model(c).to(DEVICE), ds, DEVICE, num_workers=NUM_WORKERS)
        name = f"{'bicubic' if task == 'sr' else 'zero-filled'} {task} x{f}"
        paper = PAPER_M4RAW[(task, f)]
        rows.append({"setting": name, "PSNR": round(m["psnr"], 2), "SSIM": round(m["ssim"], 4),
                     "volumes": m["n_volumes"],
                     "paper SwinIR (single-contrast)": paper["SwinIR"][0],
                     "paper SGF-MRI": paper["SGF-MRI"][0], "sec": round(time.time() - t0)})
        append_result({"run": f"baseline_{task}{f}", "checkpoint": "-", "task": task, "factor": f,
                       "psnr": m["psnr"], "psnr_std": m["psnr_std"], "ssim": m["ssim"],
                       "ssim_std": m["ssim_std"]})
    display(pd.DataFrame(rows))

,setting,PSNR,SSIM,volumes,paper SwinIR (single-contrast),paper SGF-MRI,sec
0,bicubic sr x2,27.36,0.6724,90,32.16,32.42,284
1,bicubic sr x4,23.62,0.4341,90,29.73,30.62,17
2,zero-filled recon x4,25.56,0.6518,90,29.12,29.82,12
3,zero-filled recon x8,22.94,0.5273,90,27.93,28.27,12


In [6]:
if DO_SMOKE:
    import shutil
    smoke = Config(**{**T.asdict(cfg), "run_name": "_smoke", "epochs": 2, "val_every": 1,
                      "max_train_pairs": 4, "max_val_pairs": 2})
    shutil.rmtree(smoke.out_dir, ignore_errors=True)
    t0 = time.time()
    hist = train(Config(**{**T.asdict(smoke), "epochs": 1}), CACHE_DIR)     # epoch 0
    hist = train(smoke, CACHE_DIR)                                           # resumes -> epoch 1
    assert len(hist) == 2 and hist[0]["epoch"] == 0 and hist[1]["epoch"] == 1, "resume failed"
    if hist[1]["train_l1"] >= hist[0]["train_l1"]:
        print("WARNING: loss did not go down between the two tiny epochs - rerun once; if it repeats, report it")
    n_it = len(T.make_dataset(CACHE_DIR, "train", smoke, 4)) // smoke.batch_size
    sec_it = sum(h["train_min"] for h in hist) * 60 / (2 * n_it)
    full_it = len(T.make_dataset(CACHE_DIR, "train", cfg)) // cfg.batch_size
    print(f"\nsmoke test OK in {time.time()-t0:.0f}s | ~{sec_it:.3f} s/iter "
          f"(first iterations are slower, so this is pessimistic)")
    print(f"estimate for the real run: {full_it} it/epoch -> ~{full_it*sec_it/60:.1f} min/epoch, "
          f"~{full_it*sec_it*cfg.epochs/3600:.1f} h for {cfg.epochs} epochs (+ evaluation)")
    shutil.rmtree(smoke.out_dir, ignore_errors=True)

unet_single | SR x4 | params 1.93 M | device cuda x2 | train 72 slices (18 it/epoch) | test 36 slices
  epoch   0 iter     0/18 loss 0.7809
epoch   0 | L1 0.6906 | 0.2 min | test PSNR 8.38 SSIM 0.3759 (best 8.38)
[resume] continuing from epoch 1 (best PSNR so far 8.383)
unet_single | SR x4 | params 1.93 M | device cuda x2 | train 72 slices (18 it/epoch) | test 36 slices
  epoch   1 iter     0/18 loss 0.5872
epoch   1 | L1 0.3236 | 0.0 min | test PSNR 17.67 SSIM 0.4251 (best 17.67)

smoke test OK in 130s | ~0.399 s/iter (first iterations are slower, so this is pessimistic)
estimate for the real run: 1728 it/epoch -> ~11.5 min/epoch, ~9.6 h for 50 epochs (+ evaluation)


In [7]:
if DO_TRAIN:
    history = train(cfg, CACHE_DIR)

In [8]:
if DO_EVAL and (cfg.out_dir / "last.pt").exists():
    import matplotlib.pyplot as plt
    ds = T.make_dataset(CACHE_DIR, "val", cfg)
    rows = []
    for which in ["best", "last"]:
        if not (cfg.out_dir / f"{which}.pt").exists():
            continue
        model, ep = load_for_eval(cfg, which, DEVICE)
        m, ex = evaluate(model, ds, DEVICE, num_workers=NUM_WORKERS, keep_examples=3)
        rows.append({"checkpoint": f"{which} (epoch {ep})", "PSNR": f"{m['psnr']:.2f} ± {m['psnr_std']:.2f}",
                     "SSIM": f"{m['ssim']:.4f} ± {m['ssim_std']:.3f}", "volumes": m["n_volumes"]})
        append_result({"run": cfg.run_name, "checkpoint": which, "epoch": ep, "task": cfg.task,
                       "factor": paper_key(cfg)[1], "model": cfg.model, **{k: m[k] for k in
                       ("psnr", "psnr_std", "ssim", "ssim_std")}})
    for name, (p, s) in PAPER_M4RAW[paper_key(cfg)].items():
        rows.append({"checkpoint": f"paper: {name}", "PSNR": p, "SSIM": s, "volumes": 90})
    display(pd.DataFrame(rows))
    print("efficiency:", {k: round(v, 4) for k, v in profile(model, cfg).items()})

    fig, axes = plt.subplots(len(ex), 5, figsize=(20, 4 * len(ex)), squeeze=False)
    for r, e in zip(axes, ex):
        hw = e["target"].shape
        inp = e["input"] if e["input"].shape == hw else np.kron(e["input"], np.ones((hw[0] // e["input"].shape[0],) * 2))
        vmax = e["target"].max()
        for a, img, t in zip(r, [e["ref"], inp, e["output"], e["target"], np.abs(e["output"] - e["target"])],
                             ["T1 reference", "input (degraded T2)", "model output", "ground truth T2", "|error| x5"]):
            own_scale = t in ("T1 reference", "input (degraded T2)")   # LR input is x-scale brighter
            a.imshow(img * (5 if t.startswith("|") else 1), cmap="gray",
                     vmin=None if own_scale else 0, vmax=None if own_scale else vmax)
            a.set_title(f"{t}\n{e['name']}" if t == "model output" else t); a.axis("off")
    plt.tight_layout(); plt.show()

In [9]:
log = cfg.out_dir / "log.csv"
if log.exists():
    import matplotlib.pyplot as plt
    h = pd.read_csv(log)
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot(h.epoch, h.train_l1); ax[0].set(title="training L1 loss", xlabel="epoch")
    if "psnr" in h:
        v = h.dropna(subset=["psnr"]); ax[1].plot(v.epoch, v.psnr, "o-")
        ax[1].axhline(PAPER_M4RAW[paper_key(cfg)]["SwinIR"][0], ls="--", c="gray", label="paper SwinIR")
        ax[1].set(title="test PSNR", xlabel="epoch"); ax[1].legend()
    plt.show()
if Path("/kaggle/working/results.csv").exists():
    display(pd.read_csv("/kaggle/working/results.csv"))

,run,checkpoint,task,factor,psnr,psnr_std,ssim,ssim_std
0,baseline_sr2,-,sr,2,27.3559,0.5197,0.6724,0.0170
1,baseline_sr4,-,sr,4,23.6240,0.4861,0.4341,0.0146
2,baseline_recon4,-,recon,4,25.5603,0.4937,0.6518,0.0178
3,baseline_recon8,-,recon,8,22.9363,0.4966,0.5273,0.0198
